# Fine-tuning eficiente com QLoRA — Roteador de sentimentos EscutIA

Uso rápido: selecione uma GPU no Colab e clique em **Runtime > Run all**. O notebook faz o clone, instala as dependências, valida o dataset e inicia o QLoRA.

O treinamento fica ativado por padrão. Se quiser somente preparar e validar o ambiente, mude `EXECUTE_TRAINING` para `False` antes de executar tudo.

## O que será demonstrado

- diferença entre LoRA e QLoRA;
- quantização do modelo-base em 4 bits com NF4;
- treino de um adapter pequeno sobre um modelo de 1,5B;
- reutilização de um dataset validado sem copiar dados para esta pasta;
- preservação de configuração, revisão do modelo, gate do dataset e artefatos do experimento.

O modelo continua sendo um roteador: a saída esperada é um JSON com `sentimento`, não uma resposta conversacional final.

In [ ]:
# Verifique no Colab: Runtime > Change runtime type > GPU
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memória total (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    raise RuntimeError('GPU NVIDIA/CUDA não encontrada. Ative uma GPU no runtime do Colab antes de continuar.')

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/michaeldouglas/alura-llama-factory.git'
REPO_REF = 'feature/parte-2-fine-tuning-qlora'
REPO_DIR = Path('/content/alura-llama-factory')

# A branch precisa estar publicada no GitHub antes de executar esta célula.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print(f'Repositório já existe em {REPO_DIR}; clone não repetido.')

PROJECT_DIR = REPO_DIR / 'EscutIA'
QLORA_DIR = PROJECT_DIR / 'fine_tuning_qlora'
DATASET_DIR = PROJECT_DIR / 'dataset' / 'dados' / 'preparados'
GATE_PATH = PROJECT_DIR / 'dataset' / 'dados' / 'relatorios' / '11_validacao_final.json'
CONFIG_PATH = QLORA_DIR / 'configs' / 'qlora_escutia.yaml'
OUTPUT_DIR = QLORA_DIR / 'outputs' / 'resultados' / 'qlora_escutia_router'

required_paths = [DATASET_DIR / name for name in [
    'dataset_info.json', 'escutia_train.json', 'escutia_validation.json', 'escutia_evaluation.json'
]] + [GATE_PATH, CONFIG_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Arquivos obrigatórios ausentes após o clone:\n- ' + '\n- '.join(missing))

os.chdir(QLORA_DIR)
print('Projeto:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Configuração:', CONFIG_PATH)
print('Saída:', OUTPUT_DIR)

In [ ]:
# Instala dependências específicas sem substituir o PyTorch CUDA fornecido pelo Colab.
!pip install -q -r /content/alura-llama-factory/EscutIA/fine_tuning_qlora/requirements-colab.txt
print('Dependências instaladas. Se o Colab solicitar, reinicie o runtime e reexecute as células de verificação.')

In [ ]:
import json
from collections import Counter

gate = json.loads(GATE_PATH.read_text(encoding='utf-8'))
decision = gate.get('decisao')
print('Gate do dataset:', decision)
if decision != 'DATA_READY_FOR_SFT':
    raise RuntimeError('Treinamento bloqueado: o dataset não está marcado como DATA_READY_FOR_SFT.')

def load_json_rows(path):
    value = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(value, list) or not value:
        raise ValueError(f'Dataset vazio ou inválido: {path}')
    return value

def label(row):
    output = row.get('output', {})
    if isinstance(output, str):
        output = json.loads(output)
    value = output.get('sentimento')
    if value not in {'positivo', 'neutro', 'negativo'}:
        raise ValueError(f'Rótulo inválido: {value!r}')
    return value

train_rows = load_json_rows(DATASET_DIR / 'escutia_train.json')
validation_rows = load_json_rows(DATASET_DIR / 'escutia_validation.json')
evaluation_rows = load_json_rows(DATASET_DIR / 'escutia_evaluation.json')
print('Treino:', len(train_rows), Counter(label(row) for row in train_rows))
print('Validação:', len(validation_rows), Counter(label(row) for row in validation_rows))
print('Avaliação congelada:', len(evaluation_rows), Counter(label(row) for row in evaluation_rows))
print('Arquivos preparados:', sorted(path.name for path in DATASET_DIR.iterdir()))

In [ ]:
# Inspeção final do YAML antes de qualquer execução.
import yaml

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
required_config = {
    'model_name_or_path', 'model_revision', 'quantization_method', 'quantization_bit',
    'quantization_type', 'double_quantization', 'stage', 'finetuning_type',
    'dataset_dir', 'dataset', 'eval_dataset', 'template', 'output_dir'
}
missing_config = sorted(required_config - set(config))
if missing_config:
    raise ValueError(f'Campos ausentes na configuração: {missing_config}')
if config['quantization_bit'] != 4 or config['quantization_method'] != 'bnb':
    raise ValueError('A configuração desta etapa deve usar bitsandbytes em 4 bits.')
if config['finetuning_type'] != 'lora' or config['stage'] != 'sft':
    raise ValueError('A etapa esperada é SFT com adapter LoRA.')
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f'Saída já contém artefatos; não sobrescrever: {OUTPUT_DIR}')

print(CONFIG_PATH.read_text(encoding='utf-8'))
print('Configuração validada sem iniciar treinamento.')

## Execução

A célula inicia o treinamento automaticamente quando `EXECUTE_TRAINING` está como `True`. Use `False` se quiser somente clonar, instalar e validar.

In [ ]:
EXECUTE_TRAINING = True

if not EXECUTE_TRAINING:
    print('Treinamento desativado. O ambiente e o dataset foram somente preparados e validados.')
else:
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    command = ['llamafactory-cli', 'train', str(CONFIG_PATH)]
    print('Executando:', ' '.join(command))
    subprocess.run(command, cwd=QLORA_DIR, check=True)
    print('Treinamento concluído. Confira os artefatos em:', OUTPUT_DIR)

In [ ]:
# Registro local do experimento; execute após o treinamento.
from datetime import datetime, timezone
import hashlib

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not OUTPUT_DIR.exists():
    print('Ainda não há output para registrar.')
else:
    adapter_files = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file())
    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_name_or_path': config['model_name_or_path'],
        'model_revision': config['model_revision'],
        'finetuning_type': config['finetuning_type'],
        'quantization_method': config['quantization_method'],
        'quantization_bit': config['quantization_bit'],
        'dataset_gate': decision,
        'dataset_train_records': len(train_rows),
        'dataset_validation_records': len(validation_rows),
        'output_dir': str(OUTPUT_DIR),
        'files': [{'path': str(path.relative_to(OUTPUT_DIR)), 'sha256': sha256(path)} for path in adapter_files]
    }
    manifest_path = QLORA_DIR / 'outputs' / 'qlora_run_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Manifesto salvo em:', manifest_path)
    print('Arquivos gerados:', len(adapter_files))

## Encerramento e comparação com LoRA

Ao finalizar, compare esta execução com `fine_tuning_lora` mantendo fixos o dataset, o template, o formato da saída e o procedimento de avaliação. Registre pelo menos: memória máxima da GPU, duração, tamanho do adapter, loss de validação, acurácia/F1 no conjunto congelado e taxa de JSON válido.

O adapter QLoRA depende do modelo-base e não deve ser tratado como um modelo completo. Faça download dos artefatos gerados ou copie-os para o Google Drive antes de encerrar a sessão do Colab.